In [38]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [39]:
from langchain_tavily import TavilySearch

# web搜索工具，使用tavily作为web搜索工具
web_search = TavilySearch(
    max_results=5,
    topic="general",
)

In [40]:

from langchain.chat_models import init_chat_model
import os
base_url = os.getenv('DASHSCOPE_BASE_URL')
print(base_url)

api_key = os.getenv('DASHSCOPE_API_KEY')
print(api_key)
model = os.getenv('DASHSCOPE_MODEL_ID')
print(model)

# 1.初始化模型
model = init_chat_model(
    model=model,  # 这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=base_url,
    api_key=api_key
)

https://coding.dashscope.aliyuncs.com/v1
sk-sp-31174fd20a2b4775ac6ee25a4ee8a347
qwen3.5-plus


In [41]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
import os

# 确保resources目录存在
os.makedirs("resources", exist_ok=True)

# 连接sqlite
connection = sqlite3.connect("resources/personal_chief.db", check_same_thread=False)
# 初始化checkpointer
checkpointer = SqliteSaver(connection)
# 自动建表
checkpointer.setup()

In [42]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
import os

# 确保resources目录存在
os.makedirs("resources", exist_ok=True)

# 连接sqlite
connection = sqlite3.connect("resources/personal_chief.db", check_same_thread=False)
# 初始化checkpointer
checkpointer = SqliteSaver(connection)
# 自动建表
checkpointer.setup()

In [43]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

In [44]:
from langchain.messages import HumanMessage

# 准备多模态消息，图片是网络上的冰箱食物图片
multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg"},
        {"type": "text", "text": "帮我看看这些食材能做些什么？"}
    ])

config = {"configurable": {"thread_id": "6"}}

response = agent.invoke({"messages": [multimodal_message]}, config)

In [45]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_4f27ebae620d4b15a8f118b7)
 Call ID: call_4f27ebae620d4b15a8f118b7
  Args:
    query: 生菜 彩椒 蘑菇 小番茄 西兰花 三文鱼 鸡胸肉 白萝卜 可做的菜谱
   

In [46]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第1道菜，可以说的更详细点吗？")]},
    config
)

# 友好打印
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

### 🍽️ 香煎三文鱼配时蔬沙拉 - 详细食谱

---

#### **一、食材准备**

| 类别 | 食材 | 用量 |
|------|------|------|
| **主料** | 三文鱼排 | 2块（约300g） |
| **沙拉蔬菜** | 生菜 | 1颗（撕成小块） |
| | 彩椒（红/黄） | 各半个（切丝） |
| | 小番茄 | 8-10颗（对半切） |
| | 蘑菇 | 5-6个（切片） |
| **调味料** | 橄榄油 | 2汤匙 |
| | 盐 | 适量 |
| | 黑胡椒 | 适量 |
| | 柠檬 | 半个（挤汁用） |
| | 蒜末 | 1瓣（可选） |

---

#### **二、制作步骤**

##### **步骤1：处理三文鱼** ⏱️ 5分钟
1. 三文鱼用厨房纸吸干表面水分
2. 两面均匀撒上盐和黑胡椒，腌制5分钟
3. 挤上少许柠檬汁去腥提鲜

##### **步骤2：准备沙拉蔬菜** ⏱️ 5分钟
1. 生菜洗净撕成适口大小，沥干水分
2. 彩椒去籽切细丝
3. 小番茄对半切开
4. 蘑菇切片后快速焯水30秒（去除土腥味）
5. 所有蔬菜放入大碗中混合

##### **步骤3：煎三文鱼** ⏱️ 8分钟
1. 平底锅中火预热，加入1汤匙橄榄油
2. 三文鱼**皮朝下**先煎，煎3-4分钟至皮脆
3. 翻面再煎2-3分钟（根据厚度调整）
4. 煎至表面金黄、内部呈粉红色即可
5. 出锅前加入蒜末增香（可选）

##### **步骤4：组装摆盘** ⏱️ 2分钟
1. 蔬菜沙拉铺在盘底
2. 三文鱼切块或整块放在沙拉上
3. 淋上少许橄榄油和柠檬汁
4. 最后撒少许黑胡椒装饰

---

#### **三、烹饪技巧💡**

| 技巧 | 说明 |
|------|------|
| 🔥 **火候控制** | 中火煎制，避免外焦里生 |
| 🍋 **去腥秘诀** | 柠檬汁+黑胡椒是最佳组合 |
| 🥗 **沙拉脆爽** | 蔬菜洗净后一定要沥干水分 |
| ⏰ **熟度判断** | 用叉子轻拨，鱼肉能轻松分离即熟 |
| 🧂 **调味时机** | 

运行项目
uv run langgraph dev
